# SDG 3 — Person B: Feature Engineering, Baseline Models & Experiments


## How This Notebook Connects to Person A

Person A's script (`personA_eda_preprocessing.py`) produced two clean output files:
```
devex_train_clean.csv   ← training data with a new 'clean_text' column already preprocessed
devex_test_clean.csv    ← test data with a new 'clean_text' column already preprocessed
```

Person A's preprocessing pipeline already handled:
- ✅ HTML tag and entity removal
- ✅ Lowercasing
- ✅ Regex tokenization
- ✅ Stop word removal (NLTK English list)
- ✅ Lemmatization (WordNetLemmatizer)
- ✅ Domain acronym preservation (SDG, WHO, HIV, TB, USAID, etc.)
- ✅ Numeric-only token removal
- ✅ Label binarization into a label matrix

**Person B picks up from here.** We load the clean CSV files and go straight into feature engineering.

---

## 📌 What Person B Does
```
SECTION 0  — Install & Import
SECTION 1  — Load Person A's Cleaned Outputs
SECTION 2  — Reconstruct Labels & Split Data
SECTION 3  — Feature Engineering (Text → Numbers)
             3A: Bag of Words (BoW)
             3B: TF-IDF
SECTION 4  — Evaluation Metrics (defined once, reused everywhere)
SECTION 5  — Experiments
             Exp 1: LR + TF-IDF Unigrams  (Baseline)
             Exp 2: LR + TF-IDF Bigrams
             Exp 3: Vocabulary Size Tuning
             Exp 4: Model Comparison (LR vs SVM vs RF)
             Exp 5: Threshold Optimization
             Exp 6: Class Imbalance — class_weight='balanced'
SECTION 6  — Full Results Table + Visualizations
SECTION 7  — Final Test Predictions
```

---
# SECTION 0 — Install & Import Libraries

In [ ]:
# Install libraries not pre-installed in Colab
# The '!' means: run this as a terminal command, not Python
!pip install scikit-learn pandas numpy matplotlib seaborn --quiet

In [ ]:
# ── Data & math ───────────────────────────────────────────────────────────────
import pandas as pd        # loading and manipulating tables of data
import numpy as np         # numerical operations and arrays
import warnings
warnings.filterwarnings('ignore')

# ── Plotting ──────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Scikit-learn: text feature extraction ─────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
# TfidfVectorizer → converts text into TF-IDF weighted number vectors
# CountVectorizer → converts text into raw word-count vectors (Bag of Words)

# ── Multi-label wrapper ───────────────────────────────────────────────────────
from sklearn.multiclass import OneVsRestClassifier
# OneVsRestClassifier wraps any binary classifier to handle multiple labels.
# It trains ONE separate yes/no model per label:
#   "Is this about 3.1? Yes/No"
#   "Is this about 3.2? Yes/No" ... for every SDG 3 indicator
# All models run independently and their outputs combine for the final answer.

# ── The three baseline classifiers ────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
# Draws a mathematical boundary between classes using probabilities.
# Fast, interpretable, and strong baseline for text classification.

from sklearn.svm import LinearSVC
# Support Vector Machine. Finds the widest margin between classes.
# Often excellent on high-dimensional sparse text data.

from sklearn.ensemble import RandomForestClassifier
# Builds hundreds of decision trees and takes a majority vote.
# More robust but slower and memory-heavy.

# ── Data utilities ────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
# Splits data into training set and validation set

# ── Evaluation metrics ────────────────────────────────────────────────────────
from sklearn.metrics import (
    hamming_loss,          # PRIMARY metric: fraction of wrong label predictions
    f1_score,              # balance between precision and recall
    accuracy_score,        # exact match: ALL labels must be right
    classification_report  # full per-label breakdown
)

print("All libraries imported successfully")

---
# SECTION 1 — Load Person A's Cleaned Outputs

Person A's script saved two ready-to-use files:
- `devex_train_clean.csv` — original training data + a new `clean_text` column
- `devex_test_clean.csv`  — original test data  + a new `clean_text` column

The `clean_text` column already has all preprocessing applied.  
We use that column directly — no need to redo any of Person A's work.

In [ ]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
# This gives Colab access to files you uploaded to Google Drive
from google.colab import drive
drive.mount('/content/drive')

# After this runs, your Drive files live at: /content/drive/MyDrive/

In [ ]:
# ── File paths ────────────────────────────────────────────────────────────────
# Point these to wherever you placed Person A's output files in your Drive
TRAIN_CLEAN_PATH = '/content/drive/MyDrive/devex_train_clean.csv'
TEST_CLEAN_PATH  = '/content/drive/MyDrive/devex_test_clean.csv'

# Also load Person A's label frequency summary — useful for our imbalance analysis
LABEL_FREQ_PATH  = '/content/drive/MyDrive/outputs/label_frequencies.csv'

# ── Load the files ────────────────────────────────────────────────────────────
train_df = pd.read_csv(TRAIN_CLEAN_PATH, encoding='latin-1', low_memory=False)
# encoding='latin-1' matches what Person A used in load_csv()
# low_memory=False prevents dtype guessing errors on large files

test_df  = pd.read_csv(TEST_CLEAN_PATH,  encoding='latin-1', low_memory=False)

print(f"Training rows: {len(train_df):,}")
print(f"Test rows:     {len(test_df):,}")
print(f"\nTraining columns ({len(train_df.columns)}):")
print(train_df.columns.tolist())

In [ ]:
# ── Verify clean_text column exists ──────────────────────────────────────────
# Person A's preprocess_text() function saved results into 'clean_text'
assert 'clean_text' in train_df.columns, \
    "ERROR: 'clean_text' column not found. Make sure you're loading Person A's OUTPUT file (devex_train_clean.csv), not the original."

assert 'clean_text' in test_df.columns, \
    "ERROR: 'clean_text' column not found in test file."

# Preview: compare raw text vs Person A's cleaned text side by side
# This shows exactly what Person A's pipeline did
raw_col = [c for c in train_df.columns if 'text' in c.lower() and c != 'clean_text'][0]
# Find the original raw text column (the one Person A's detect_text_column() found)

print("=== COMPARISON: Raw text vs Person A's clean_text ===\n")
for i in range(2):
    print(f"--- Document {i} ---")
    print(f"RAW:   {str(train_df[raw_col].iloc[i])[:300]}")
    print(f"CLEAN: {str(train_df['clean_text'].iloc[i])[:300]}")
    print()
    
print("✅ clean_text column confirmed. Person A's preprocessing is intact.")
RAW_TEXT_COL = raw_col  # save for reference

In [ ]:
# ── Handle any remaining missing values in clean_text ────────────────────────
# Person A's normalize_text() handles empty strings, but we double-check
missing_train = train_df['clean_text'].isna().sum()
missing_test  = test_df['clean_text'].isna().sum()
print(f"Missing clean_text values — train: {missing_train}, test: {missing_test}")

# Fill any NaN with empty string (safe fallback)
train_df['clean_text'] = train_df['clean_text'].fillna('')
test_df['clean_text']  = test_df['clean_text'].fillna('')

# Drop rows where clean_text ended up empty after preprocessing
# (these would be documents that had no usable tokens)
before = len(train_df)
train_df = train_df[train_df['clean_text'].str.strip() != ''].reset_index(drop=True)
# .str.strip() removes leading/trailing whitespace
# We keep only rows where the result is NOT an empty string
print(f"Dropped {before - len(train_df)} empty documents. Remaining: {len(train_df):,}")

---
# SECTION 2 — Reconstruct Labels & Split Data

Person A used `detect_label_columns()` and `detect_label_format()` to identify the label columns.  
Their `build_label_lists()` and `MultiLabelBinarizer` converted them into a binary matrix.

We replicate that exact logic here so our label matrix matches Person A's perfectly.

In [ ]:
# ── Re-detect label columns using Person A's same logic ──────────────────────
# Person A's detect_label_columns() looked for columns whose values match
# the pattern 'digit.digit' (e.g., '3.1', '3.2') in at least 50% of rows.
# We replicate this exactly.

import re

def detect_label_columns(df, text_col):
    # First try: columns explicitly named 'label'
    label_cols = [c for c in df.columns if 'label' in c.lower()]
    if label_cols:
        return label_cols
    
    # Second try: columns whose values look like SDG indicators (e.g., '3.1')
    label_like = []
    for col in df.columns:
        if col == text_col or col == 'clean_text':
            continue
        if df[col].dtype not in ['object', 'string', 'str']:
            continue
        sample = df[col].dropna().astype(str).head(200)
        if sample.empty:
            continue
        sdg_ratio = sample.str.contains(r'\d+(\.\d+)+', regex=True).mean()
        # Checks if the value matches patterns like '3.1', '3.2b' etc.
        if sdg_ratio >= 0.5:
            label_like.append(col)
    return label_like

LABEL_COLS = detect_label_columns(train_df, RAW_TEXT_COL)
print(f"Detected label columns ({len(LABEL_COLS)}): {LABEL_COLS}")

In [ ]:
# ── Detect label format ───────────────────────────────────────────────────────
# Person A's detect_label_format() distinguished between:
#   'binary'       → label columns already contain 0s and 1s
#   'multi-column' → label columns contain strings like '3.1', '3.2'

def detect_label_format(df, label_cols):
    if not label_cols:
        return 'none'
    numeric_cols = [c for c in label_cols if pd.api.types.is_numeric_dtype(df[c])]
    if len(numeric_cols) == len(label_cols):
        unique_vals = set(pd.unique(df[label_cols].values.ravel()))
        unique_vals = {v for v in unique_vals if not pd.isna(v)}
        if unique_vals.issubset({0, 1}):
            return 'binary'
    return 'multi-column'

LABEL_FORMAT = detect_label_format(train_df, LABEL_COLS)
print(f"Label format: '{LABEL_FORMAT}'")
# 'binary'       → columns are already 0/1, we use them directly as our Y matrix
# 'multi-column' → we need MultiLabelBinarizer to convert them

In [ ]:
# ── Build the label matrix Y ──────────────────────────────────────────────────
# Y is a 2D array of 0s and 1s: shape = (n_documents, n_labels)
# Y[i][j] = 1 means document i is related to label j

from sklearn.preprocessing import MultiLabelBinarizer

if LABEL_FORMAT == 'binary':
    # Labels are already 0/1 — use directly
    Y = train_df[LABEL_COLS].values.astype(int)
    # .values converts the pandas DataFrame columns into a numpy array
    # .astype(int) ensures values are integers not floats
    LABEL_NAMES = LABEL_COLS
    # Label names are the column names themselves

else:
    # Labels are strings like '3.1', '3.2' — binarize them
    # Mirrors exactly what Person A's build_label_lists() + mlb.fit_transform() did
    labels_list = [
        [str(v).strip() for v in row
         if pd.notna(v) and str(v).strip() and str(v).strip().upper() != 'NA']
        for row in train_df[LABEL_COLS].values.tolist()
    ]
    # labels_list is a list of lists:
    # e.g., [['3.1', '3.3'], ['3.2'], ['3.1', '3.2', '3.5'], ...]

    mlb = MultiLabelBinarizer()
    Y = mlb.fit_transform(labels_list)
    # fit_transform() learns all unique labels and converts each list to a binary row
    # e.g., ['3.1', '3.3'] → [1, 0, 1, 0, 0, 0, ...] (based on sorted unique labels)
    LABEL_NAMES = list(mlb.classes_)
    # mlb.classes_ gives the sorted unique label names corresponding to each column

print(f"Y shape: {Y.shape}")
# Expected: (n_documents, n_labels) — e.g., (3000, 13)
print(f"Number of unique labels: {len(LABEL_NAMES)}")
print(f"Label names: {LABEL_NAMES}")
print(f"\nY sample (first document):")
print(f"  Labels: {Y[0]}")
print(f"  Active: {[LABEL_NAMES[i] for i, v in enumerate(Y[0]) if v == 1]}")

In [ ]:
# ── Load Person A's label frequency summary ───────────────────────────────────
# Person A already computed and saved this — we just load it
# This feeds our understanding of class imbalance for Experiments 5 & 6

try:
    label_freq_df = pd.read_csv(LABEL_FREQ_PATH)
    print("Person A's label frequency summary:")
    print(label_freq_df.to_string(index=False))
    
    most_common = label_freq_df.sort_values('count', ascending=False).head(3)['label'].tolist()
    rarest      = label_freq_df.sort_values('count', ascending=True).head(3)['label'].tolist()
    print(f"\n⚠ Most common labels (high frequency): {most_common}")
    print(f"⚠ Rarest labels (risk of being ignored): {rarest}")
    print("→ This imbalance directly motivates Experiment 6 (class_weight='balanced')")

except FileNotFoundError:
    print("Label frequency CSV not found — computing from Y matrix instead.")
    label_counts = Y.sum(axis=0)
    label_freq_df = pd.DataFrame({'label': LABEL_NAMES, 'count': label_counts})
    label_freq_df = label_freq_df.sort_values('count', ascending=False)
    print(label_freq_df.to_string(index=False))

In [ ]:
# ── Train / Validation Split ───────────────────────────────────────────────────
# We split data into:
# - TRAIN SET (80%): model learns from this
# - VALIDATION SET (20%): we measure performance on this
#
# CRITICAL: Never evaluate on the same data you trained on.
# The model would just memorize all the training answers — like giving
# a student the exam answer sheet before the test. It would score 100%
# but learn nothing about new documents.
#
# We use clean_text (Person A's preprocessed text) as our X input.

X_raw   = train_df['clean_text'].values
# X_raw = array of preprocessed text strings — ready for vectorization

X_train, X_val, Y_train, Y_val = train_test_split(
    X_raw,
    Y,
    test_size=0.2,      # 20% goes to validation
    random_state=42,    # fixed seed = same split every time = reproducible results
)

print(f"Training samples:   {len(X_train):,}")
print(f"Validation samples: {len(X_val):,}")
print(f"Y_train shape: {Y_train.shape}")
print(f"Y_val shape:   {Y_val.shape}")

---
# SECTION 3 — Feature Engineering: Converting Text to Numbers

**The fundamental problem:** Machine learning models are purely mathematical.  
They cannot read words. We must convert every document into a **vector** (a list of numbers).

Person A already cleaned the text. Now Person B converts it into numbers.

We implement two methods:
1. **Bag of Words (BoW)** — raw word counts
2. **TF-IDF** — smarter weighted scores that reward rare, informative words

---
## 3A — Bag of Words (BoW)

**How it works:**
1. Build a vocabulary of every unique word across all documents
2. For each document, count how many times each vocabulary word appears
3. That vector of counts = the document's numerical representation

Example with vocabulary `[health, malaria, school, water]`:
```
Document: "health programs fight malaria and improve health outcomes"
BoW:      [2,      1,       0,      0    ]
           ↑       ↑        ↑       ↑
         health  malaria  school  water
```

**Why 'bag'?** It throws away word ORDER.  
*"HIV treatment reduces mortality"* and *"mortality reduces HIV treatment"* are identical in BoW.  
The 'bag' just holds words without caring about their position.

In [ ]:
# ── Bag of Words Vectorizer ───────────────────────────────────────────────────
bow_vectorizer = CountVectorizer(
    max_features=10000,
    # Keep only the 10,000 most frequent words.
    # Our cleaned corpus may still have 30,000+ unique tokens.
    # Rare words (appearing once or twice) add noise, not signal — we cut them.

    # NOTE: We do NOT pass stop_words here because Person A already removed them.
    # Double-removing stop words is harmless but redundant.
    # Person A used NLTK's English stop word list in preprocess_text().

    min_df=2,
    # Ignore any word appearing in fewer than 2 documents.
    # 'df' = document frequency = how many documents contain this word.
    # Words in only 1 document = likely a typo or ultra-niche term.

    max_df=0.95,
    # Ignore words appearing in MORE than 95% of documents.
    # Such words appear everywhere and don't help distinguish topics.
    # After Person A removed stopwords, remaining high-freq words like
    # 'program', 'project', 'health' may still dominate — we cap them.

    lowercase=False,
    # Person A already lowercased everything EXCEPT domain acronyms.
    # e.g., 'HIV', 'WHO', 'TB' were kept uppercase intentionally.
    # Setting lowercase=False respects Person A's decision to preserve these.
    # If we set True, 'HIV' becomes 'hiv' and loses its acronym identity.
)

# ── Fit on TRAINING data only, then transform both sets ──────────────────────
# fit_transform() = step 1: scan training docs to learn vocabulary
#                   step 2: convert training docs to count vectors
# We ONLY fit on training data — NEVER validation or test data.
# Reason: if we fit on all data, the model gets advance knowledge of
# test vocabulary = 'data leakage' = falsely inflated performance.
X_train_bow = bow_vectorizer.fit_transform(X_train)

# transform() = step 2 only (vocabulary already fixed from training)
# Words in validation that don't exist in training vocab are simply ignored.
X_val_bow   = bow_vectorizer.transform(X_val)

print(f"BoW matrix shape (train): {X_train_bow.shape}")
# (n_train_docs, vocabulary_size) — e.g., (2400, 10000)
print(f"BoW matrix shape (val):   {X_val_bow.shape}")
print(f"Actual vocabulary size:   {len(bow_vectorizer.vocabulary_):,}")

# Check how sparse the matrix is (most cells will be 0)
total_cells = X_train_bow.shape[0] * X_train_bow.shape[1]
nonzero     = X_train_bow.nnz  # nnz = number of non-zero entries
sparsity    = 1 - (nonzero / total_cells)
print(f"Matrix sparsity: {sparsity:.1%} zeros")
# Very high sparsity (99%+) is completely normal for text data.
# Each document only uses a tiny fraction of the entire vocabulary.

---
## 3B — TF-IDF (Term Frequency–Inverse Document Frequency)

**The problem with raw BoW counts:**  
The word *'health'* appears in almost every SDG 3 document.  
BoW gives it a high count — but it doesn't actually distinguish one document from another.

**TF-IDF fixes this** by rewarding words that are frequent in ONE document but rare across ALL documents:

```
TF-IDF(word, document) = TF × IDF

TF  = (times word appears in this doc) / (total words in this doc)
     → How prominent is this word IN THIS specific document?

IDF = log( total_documents / documents_containing_word )
     → How RARE and therefore INFORMATIVE is this word?
     → Rare word → high IDF → more weight
     → Common word → low IDF → less weight
```

**Concrete example:**
- *'onchocerciasis'* (river blindness) appears in 3 documents → very high IDF → high TF-IDF
- *'health'* appears in 2900 documents → very low IDF → low TF-IDF despite high count

TF-IDF is the standard starting point for text classification and almost always outperforms raw BoW.

In [ ]:
# ── TF-IDF Vectorizer (Experiment 1 baseline configuration) ──────────────────
tfidf_vectorizer = TfidfVectorizer(
    max_features=10000,   # top 10,000 words (same starting point as BoW)
    min_df=2,             # ignore words in fewer than 2 documents
    max_df=0.95,          # ignore words in more than 95% of documents
    lowercase=False,      # respect Person A's acronym preservation (HIV, WHO, etc.)

    ngram_range=(1, 1),
    # What word combinations to include.
    # (1,1) = unigrams only: individual words — 'maternal', 'mortality', 'HIV'
    # (1,2) = unigrams + bigrams: also word pairs — 'maternal mortality', 'HIV treatment'
    # We start with (1,1) as the baseline. Experiment 2 will test (1,2).

    sublinear_tf=True,
    # Instead of raw TF count, use: 1 + log(count)
    # This compresses the difference between a word appearing 5 times vs 50 times.
    # Without this, a word appearing 50x is 10x more influential than 5x —
    # that exaggerates repetition. With log scaling: 5→1.7, 50→2.7 (less extreme).
    # Generally improves classification performance on text data.

    analyzer='word',
    # Split text into words (not characters).
    # 'word' is the default and correct choice for English text classification.
)

# Fit on training, transform both sets (same rules as BoW)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf   = tfidf_vectorizer.transform(X_val)

print(f"TF-IDF matrix shape (train): {X_train_tfidf.shape}")
print(f"TF-IDF matrix shape (val):   {X_val_tfidf.shape}")

# ── Show most informative words (highest IDF = rarest, most distinctive) ─────
vocab      = tfidf_vectorizer.vocabulary_  # word → column index
idf_scores = tfidf_vectorizer.idf_          # IDF score for each column

top_idf = sorted(vocab.items(), key=lambda x: idf_scores[x[1]], reverse=True)[:20]
print("\nTop 20 most distinctive words by IDF (rarest across the corpus):")
for word, idx in top_idf:
    print(f"  {word:<35} IDF = {idf_scores[idx]:.4f}")

# ── Show words that TF-IDF DOWN-weights (common across all docs) ──────────────
bottom_idf = sorted(vocab.items(), key=lambda x: idf_scores[x[1]])[:10]
print("\nBottom 10 least distinctive words (most common, lowest IDF):")
for word, idx in bottom_idf:
    print(f"  {word:<35} IDF = {idf_scores[idx]:.4f}")

---
# SECTION 4 — Evaluation Metrics

Before running any experiments, we define **how we measure success**.  
This function is written once and called after every experiment.

## Primary Metric: Hamming Loss *(specified by the assignment)*

```
Hamming Loss = wrong_label_predictions / (n_documents × n_labels)
```

Example:
```
True labels:      [1, 0, 1, 0, 1]  ← correct answers
Predicted labels: [1, 1, 1, 0, 0]  ← model's predictions
                       ↑        ↑
                    WRONG    WRONG  → 2 errors out of 5
Hamming Loss = 2/5 = 0.40
```
- **0.0** = perfect
- **1.0** = every label wrong
- **Lower is always better**

## Secondary Metrics
| Metric | What it measures | Good for |
|--------|-----------------|----------|
| **Micro F1** | Overall precision/recall pooled across all labels | Overall performance |
| **Macro F1** | Average F1 per label (equal weight per label) | Detecting poor performance on rare labels |
| **Subset Accuracy** | Fraction of docs with ALL labels exactly right | Strictest quality check |

In [ ]:
# ── Universal evaluation function ─────────────────────────────────────────────
# Write this once, call it after every experiment.

def evaluate(Y_true, Y_pred, name="Model"):
    """
    Compute and display all evaluation metrics for a multi-label classifier.

    Parameters
    ----------
    Y_true : 2D numpy array, shape (n_samples, n_labels)
        Ground-truth binary label matrix (0s and 1s)
    Y_pred : 2D numpy array, shape (n_samples, n_labels)
        Predicted binary label matrix (0s and 1s)
    name : str
        Label for display purposes

    Returns
    -------
    dict with all metric scores (for storage in results table)
    """

    # 1. Hamming Loss — PRIMARY METRIC
    hl = hamming_loss(Y_true, Y_pred)
    # Counts fraction of all (doc, label) pairs predicted wrong.
    # Lower = better.

    # 2. Micro F1 — pools ALL label predictions together
    # TP = True Positive  (predicted 1, actually 1) ✅
    # FP = False Positive (predicted 1, actually 0) ❌ 'false alarm'
    # FN = False Negative (predicted 0, actually 1) ❌ 'miss'
    # Precision = TP / (TP+FP)  → of everything we said was positive, how many were right?
    # Recall    = TP / (TP+FN)  → of everything that was actually positive, how many did we catch?
    # F1 = 2 × (Precision × Recall) / (Precision + Recall)  → harmonic mean of both
    micro_f1 = f1_score(Y_true, Y_pred, average='micro', zero_division=0)

    # 3. Macro F1 — computes F1 per label, then averages equally
    # If macro F1 << micro F1, it means rare labels are being badly predicted.
    # This is the key metric to track when addressing class imbalance.
    macro_f1 = f1_score(Y_true, Y_pred, average='macro', zero_division=0)

    # 4. Subset Accuracy — the strictest metric
    # A prediction is only correct if EVERY SINGLE label is correct.
    # One wrong label out of 13 = 0 (incorrect) for that document.
    subset_acc = accuracy_score(Y_true, Y_pred)

    print(f"\n{'═'*52}")
    print(f"  {name}")
    print(f"{'═'*52}")
    print(f"  Hamming Loss      (↓ lower=better) : {hl:.4f}")
    print(f"  Micro F1          (↑ higher=better): {micro_f1:.4f}")
    print(f"  Macro F1          (↑ higher=better): {macro_f1:.4f}")
    print(f"  Subset Accuracy   (↑ higher=better): {subset_acc:.4f}")
    print(f"{'═'*52}")

    return {
        'Experiment':       name,
        'Hamming Loss':     round(hl, 4),
        'Micro F1':         round(micro_f1, 4),
        'Macro F1':         round(macro_f1, 4),
        'Subset Accuracy':  round(subset_acc, 4),
    }


# Results log — we append one dict after each experiment
all_results = []

print("✅ Evaluation function ready. Results tracker initialized.")

---
# SECTION 5 — Experiments

Each experiment follows this exact structure:
1. **What changed** from the previous experiment
2. **Why** — what the previous result told us that motivated this change
3. **The code** — train, predict, evaluate
4. **The insight** — what we learned

This structure is what the rubric rewards. It turns a list of model runs into a real research progression.

---
## Experiment 1 — Baseline: Logistic Regression + TF-IDF (Unigrams)

**What changed:** Nothing — this is the starting point.

**Why:** Every research project needs a baseline to compare against. Logistic Regression with TF-IDF is the classic text classification baseline — fast, interpretable, and deceptively strong.

### How Logistic Regression works for multi-label classification
For each label (e.g., indicator 3.1):
1. Multiply each TF-IDF feature value by a learned weight
2. Sum everything → one score
3. Pass through sigmoid function → probability between 0 and 1
4. If probability > 0.5 → predict label = 1, else predict 0

The `OneVsRestClassifier` wrapper runs this for every label independently.

In [ ]:
# ════════════════════════════════════════════════
#  EXPERIMENT 1 — LR + TF-IDF Unigrams (Baseline)
# ════════════════════════════════════════════════
print("Running Experiment 1...")

exp1_model = OneVsRestClassifier(
    LogisticRegression(
        C=1.0,
        # C = regularization parameter (INVERSE strength).
        # Regularization prevents overfitting by discouraging extreme weights.
        # Small C → strong regularization → simpler model (may underfit).
        # Large C → weak regularization  → complex model (may overfit).
        # C=1.0 is the standard default starting point — always begin here.

        solver='lbfgs',
        # The mathematical algorithm used to find the best weights.
        # 'lbfgs' = Limited-memory BFGS optimization.
        # sklearn recommends this for medium-to-large datasets.

        max_iter=1000,
        # Maximum number of optimization steps before stopping.
        # Default (100) often isn't enough for text data with many features.
        # 1000 gives the solver enough time to converge (find the best weights).

        class_weight=None,
        # None = treat every class equally.
        # 'balanced' = weight rare classes more heavily (we test this in Experiment 6).

        random_state=42,
        # Fixes randomness → same result every run = reproducible
    ),
    n_jobs=-1
    # n_jobs=-1 = use ALL available CPU cores.
    # OneVsRest trains one model per label — they can train in parallel.
)

# .fit() trains the model: it looks at (X_train, Y_train)
# and adjusts weights until predictions are as accurate as possible
exp1_model.fit(X_train_tfidf, Y_train)

# .predict() runs the trained model on validation data it has NEVER seen
# Returns: binary 0/1 matrix — shape (n_val_docs, n_labels)
Y_pred_exp1 = exp1_model.predict(X_val_tfidf)

results_exp1 = evaluate(Y_val, Y_pred_exp1, "Exp 1: LR + TF-IDF Unigrams (Baseline)")
all_results.append(results_exp1)

# ── Inspect predictions vs truth ──────────────────────────────────────────────
print("\nSample: True vs Predicted labels for first 3 validation documents")
for i in range(3):
    true = [LABEL_NAMES[j] for j, v in enumerate(Y_val[i]) if v == 1]
    pred = [LABEL_NAMES[j] for j, v in enumerate(Y_pred_exp1[i]) if v == 1]
    print(f"  Doc {i}: TRUE={true}  PRED={pred}")

---
## Experiment 2 — Add Bigrams to TF-IDF

**What changed:** `ngram_range=(1,1)` → `ngram_range=(1,2)`

**Why:** Experiment 1 only uses individual words (unigrams). But many SDG 3 topics are meaningfully expressed as two-word phrases. Breaking these apart loses their meaning:
- *'maternal mortality'* split into 'maternal' + 'mortality' loses the specific meaning
- *'universal health coverage'* split loses the policy concept
- *'HIV treatment'* split loses the treatment context

Adding bigrams lets TF-IDF treat these pairs as single features.

**Tradeoff to watch:** Bigrams multiply the vocabulary size (more features = more memory + slower training). We need to check if the performance gain justifies the cost.

In [ ]:
# ════════════════════════════════════════════════
#  EXPERIMENT 2 — LR + TF-IDF with Bigrams
# ════════════════════════════════════════════════
print("Running Experiment 2...")

tfidf_bigram = TfidfVectorizer(
    max_features=10000,
    min_df=2,
    max_df=0.95,
    lowercase=False,     # keep Person A's acronym preservation
    sublinear_tf=True,

    ngram_range=(1, 2),
    # (1,2) = include both unigrams AND bigrams in the vocabulary.
    # Unigrams: 'maternal', 'mortality', 'health'
    # Bigrams:  'maternal mortality', 'child health', 'HIV treatment'
    # The vectorizer creates features for BOTH individual words and word pairs.
    # max_features=10000 now selects the top 10,000 from this larger combined set.
)

X_train_bigram = tfidf_bigram.fit_transform(X_train)
X_val_bigram   = tfidf_bigram.transform(X_val)

print(f"Vocabulary with unigrams only: {len(tfidf_vectorizer.vocabulary_):,}")
print(f"Vocabulary with bigrams added: {len(tfidf_bigram.vocabulary_):,}")

# Show some bigrams that were learned
bigrams_found = [f for f in tfidf_bigram.vocabulary_.keys() if ' ' in f]
print(f"\nSample domain bigrams captured (first 20):")
print(bigrams_found[:20])

exp2_model = OneVsRestClassifier(
    LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    n_jobs=-1
)
exp2_model.fit(X_train_bigram, Y_train)
Y_pred_exp2 = exp2_model.predict(X_val_bigram)

results_exp2 = evaluate(Y_val, Y_pred_exp2, "Exp 2: LR + TF-IDF Bigrams")
all_results.append(results_exp2)

diff = results_exp1['Hamming Loss'] - results_exp2['Hamming Loss']
print(f"\nHamming Loss change from Exp 1: {diff:+.4f}  ({'improvement ✅' if diff > 0 else 'worse ❌' if diff < 0 else 'no change'})") 
print("📝 Insight: Did bigrams capture domain phrases? Was the vocabulary tradeoff worth it?")

---
## Experiment 3 — Vocabulary Size Tuning

**What changed:** Systematically vary `max_features` across {5,000 / 10,000 / 20,000 / 50,000}

**Why:** We assumed `max_features=10,000` in Experiments 1 and 2. That assumption may be wrong:
- **Too small (5,000):** We might cut important but less-frequent domain terms
- **Too large (50,000):** We include rare noisy words that confuse the model

We use the better ngram configuration from Experiments 1 vs 2 and vary only vocabulary size.

In [ ]:
# ════════════════════════════════════════════════
#  EXPERIMENT 3 — Vocabulary Size Tuning
# ════════════════════════════════════════════════
print("Running Experiment 3...\n")

# Use the better ngram range from Experiments 1 vs 2
best_ngram = (1, 2) if results_exp2['Hamming Loss'] < results_exp1['Hamming Loss'] else (1, 1)
print(f"Using ngram_range={best_ngram} (winner from Exp 1 vs 2)\n")

vocab_sizes = [5000, 10000, 20000, 50000]
exp3_rows = []

for size in vocab_sizes:
    print(f"Testing max_features={size:,}...", end=' ')

    vec = TfidfVectorizer(
        max_features=size,  # ← the only thing changing each iteration
        min_df=2,
        max_df=0.95,
        lowercase=False,
        sublinear_tf=True,
        ngram_range=best_ngram,
    )
    Xtr = vec.fit_transform(X_train)
    Xvl = vec.transform(X_val)

    model = OneVsRestClassifier(
        LogisticRegression(C=1.0, max_iter=1000, random_state=42),
        n_jobs=-1
    )
    model.fit(Xtr, Y_train)
    preds = model.predict(Xvl)

    hl   = hamming_loss(Y_val, preds)
    mf1  = f1_score(Y_val, preds, average='micro', zero_division=0)
    maf1 = f1_score(Y_val, preds, average='macro', zero_division=0)
    print(f"HL={hl:.4f}  MicroF1={mf1:.4f}  MacroF1={maf1:.4f}")

    exp3_rows.append({'vocab_size': size, 'Hamming Loss': round(hl,4),
                      'Micro F1': round(mf1,4), 'Macro F1': round(maf1,4)})

exp3_df = pd.DataFrame(exp3_rows)
print("\n── Vocabulary Size Results ──")
print(exp3_df.to_string(index=False))

# Pick the best vocabulary size
best_row       = exp3_df.loc[exp3_df['Hamming Loss'].idxmin()]
BEST_VOCAB     = int(best_row['vocab_size'])
print(f"\n✅ Best vocabulary size: {BEST_VOCAB:,}  (Hamming Loss = {best_row['Hamming Loss']})")

# Record the best result
results_exp3 = {
    'Experiment':      f'Exp 3: LR + vocab={BEST_VOCAB:,}',
    'Hamming Loss':    best_row['Hamming Loss'],
    'Micro F1':        best_row['Micro F1'],
    'Macro F1':        best_row['Macro F1'],
    'Subset Accuracy': None
}
all_results.append(results_exp3)

# ── Plot vocab size vs Hamming Loss ───────────────────────────────────────────
plt.figure(figsize=(7, 4))
plt.plot(exp3_df['vocab_size'], exp3_df['Hamming Loss'],
         marker='o', color='steelblue', linewidth=2)
plt.xlabel('Vocabulary Size (max_features)')
plt.ylabel('Hamming Loss  (lower = better)')
plt.title('Experiment 3: Effect of Vocabulary Size on Hamming Loss')
plt.xscale('log')   # log scale: 5k→10k→20k→50k spaced evenly visually
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('exp3_vocab_tuning.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Saved: exp3_vocab_tuning.png")

---
## Experiment 4 — Model Comparison: LR vs SVM vs Random Forest

**What changed:** Replace Logistic Regression with two alternative classifiers

**Why:** LR established our baseline. But different algorithms make different assumptions:
- **LR** finds a probabilistic hyperplane — works well when classes are linearly separable
- **LinearSVC** finds the MAXIMUM MARGIN hyperplane — often stronger on high-dimensional sparse text data because it focuses on the hardest-to-classify examples (the 'support vectors')
- **Random Forest** builds many decision trees and votes — captures non-linear patterns but may struggle with sparse high-dimensional text

We use the best TF-IDF config from Experiment 3 for a fair comparison.

In [ ]:
# ════════════════════════════════════════════════
#  EXPERIMENT 4 — Model Comparison
# ════════════════════════════════════════════════
print("Running Experiment 4 — Model Comparison...\n")

# ── Rebuild TF-IDF with the best config from Experiment 3 ─────────────────────
tfidf_best = TfidfVectorizer(
    max_features=BEST_VOCAB,
    min_df=2,
    max_df=0.95,
    lowercase=False,
    sublinear_tf=True,
    ngram_range=best_ngram,
)
X_train_best = tfidf_best.fit_transform(X_train)
X_val_best   = tfidf_best.transform(X_val)
print(f"Using TF-IDF: vocab={BEST_VOCAB:,}, ngrams={best_ngram}\n")

# ── Define the three models ────────────────────────────────────────────────────
models = {

    "LR (best config)": OneVsRestClassifier(
        LogisticRegression(C=1.0, max_iter=1000, random_state=42),
        n_jobs=-1
    ),

    "LinearSVC": OneVsRestClassifier(
        LinearSVC(
            C=1.0,
            # Same C regularization concept as LR.
            # SVM finds the decision boundary with MAXIMUM MARGIN between classes.
            # Text data is usually linearly separable in high dimensions,
            # making SVM a natural fit.

            max_iter=2000,
            # SVM's optimization often takes more iterations to converge.

            random_state=42,
        ),
        n_jobs=-1
    ),

    "Random Forest": OneVsRestClassifier(
        RandomForestClassifier(
            n_estimators=100,
            # Number of decision trees.
            # More trees = more stable, but slower and more memory.
            # 100 is a reliable starting point.

            min_samples_split=5,
            # A node must have at least 5 samples to split further.
            # Higher values create shallower, simpler trees = less overfitting.

            n_jobs=-1,
            random_state=42,
        ),
        n_jobs=1
        # RF already parallelizes internally (n_jobs=-1 above).
        # Setting n_jobs=1 on the OvR wrapper avoids double-parallelization.
    ),
}

# ── Train and evaluate all three ──────────────────────────────────────────────
exp4_results = []
for name, model in models.items():
    print(f"Training: {name}...")
    model.fit(X_train_best, Y_train)
    preds  = model.predict(X_val_best)
    result = evaluate(Y_val, preds, f"Exp 4: {name}")
    exp4_results.append(result)
    all_results.append(result)

# ── Comparison plot ───────────────────────────────────────────────────────────
exp4_df = pd.DataFrame(exp4_results)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['steelblue', 'coral', 'seagreen']

for ax, metric in zip(axes, ['Hamming Loss', 'Micro F1', 'Macro F1']):
    short_names = [r.replace('Exp 4: ','') for r in exp4_df['Experiment']]
    ax.bar(short_names, exp4_df[metric], color=colors)
    ax.set_title(metric)
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=20)

plt.suptitle('Experiment 4: Model Comparison', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('exp4_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Saved: exp4_model_comparison.png")

winner = exp4_df.loc[exp4_df['Hamming Loss'].idxmin(), 'Experiment']
print(f"\n✅ Best model in Exp 4: {winner}")
print("📝 Insight: Why did that model win? Consider: SVM is known to excel on sparse high-dim text.")

---
## Experiment 5 — Threshold Optimization

**What changed:** Replace fixed threshold (0.5) with an optimized per-label threshold

**Why:** By default, a label is predicted as 1 if the model's probability score ≥ 0.5.  
But for rare labels, the model may only ever produce probabilities of 0.2–0.3, so it always predicts 0 for them (false negatives). We can find the threshold per label that maximizes F1 on the validation set.

**Note:** `LinearSVC` doesn't output probabilities — we use `LogisticRegression` here because `.predict_proba()` is required.

In [ ]:
# ════════════════════════════════════════════════
#  EXPERIMENT 5 — Threshold Optimization
# ════════════════════════════════════════════════
print("Running Experiment 5 — Threshold Optimization...\n")

# Train LR to get probability outputs
lr_proba = OneVsRestClassifier(
    LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    n_jobs=-1
)
lr_proba.fit(X_train_best, Y_train)

# .predict_proba() returns a probability matrix — values between 0.0 and 1.0
# Shape: (n_val_samples, n_labels)
# Y_val_proba[i][j] = probability that document i has label j
Y_val_proba = lr_proba.predict_proba(X_val_best)

print(f"Probability matrix shape: {Y_val_proba.shape}")
print(f"Sample probabilities (doc 0): {Y_val_proba[0].round(3)}")

# Baseline: default threshold of 0.5
Y_pred_default = (Y_val_proba >= 0.5).astype(int)
# (Y_val_proba >= 0.5) creates a boolean matrix: True where prob is at least 0.5
# .astype(int) converts True→1, False→0
hl_default = hamming_loss(Y_val, Y_pred_default)
print(f"\nHamming Loss with default threshold (0.5): {hl_default:.4f}")

# ── Find the optimal threshold per label ─────────────────────────────────────
# For each label, we try 50 different thresholds and keep the one with best F1
thresholds_to_try = np.linspace(0.1, 0.9, 50)
# np.linspace(0.1, 0.9, 50) creates 50 evenly spaced values from 0.1 to 0.9:
# [0.1, 0.116, 0.133, ..., 0.883, 0.9]

best_thresholds = []

for j in range(Y.shape[1]):
    # j = label index (0 = first label, 1 = second label, etc.)
    label_probs = Y_val_proba[:, j]
    # [:, j] = all rows, column j → probability scores for label j across all docs
    label_true  = Y_val[:, j]
    # True 0/1 labels for label j

    best_f1, best_t = 0, 0.5  # start with defaults

    for t in thresholds_to_try:
        preds = (label_probs >= t).astype(int)
        f1    = f1_score(label_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t

    best_thresholds.append(round(best_t, 3))

print("\nOptimal threshold per label:")
for label, thresh in zip(LABEL_NAMES, best_thresholds):
    flag = "  ← significantly different from 0.5" if abs(thresh - 0.5) > 0.15 else ""
    print(f"  {label:<12}: {thresh:.3f}{flag}")

# ── Apply optimized thresholds ────────────────────────────────────────────────
Y_pred_opt = np.zeros_like(Y_val_proba, dtype=int)
# Create a matrix of zeros with the same shape as our probability matrix

for j, t in enumerate(best_thresholds):
    Y_pred_opt[:, j] = (Y_val_proba[:, j] >= t).astype(int)
    # For each label j, apply its own optimal threshold t

results_exp5 = evaluate(Y_val, Y_pred_opt, "Exp 5: LR + Optimized Thresholds")
all_results.append(results_exp5)

print(f"\nHamming Loss: default={hl_default:.4f} → optimized={results_exp5['Hamming Loss']:.4f}")
print("📝 Insight: Labels with thresholds far from 0.5 reveal which labels are hard to calibrate.")

---
## Experiment 6 — Class Imbalance: `class_weight='balanced'`

**What changed:** Add `class_weight='balanced'` to Logistic Regression

**Why:** Person A's EDA (loaded in Section 2) showed significant label imbalance — some SDG 3 indicators appear in far fewer documents than others. The model barely sees rare labels during training, so it learns to predict 0 for them almost always. This hurts Macro F1 (which equally weights all labels).

`class_weight='balanced'` tells the model: *"a mistake on a rare label should be penalized more heavily than a mistake on a common label."*

The weight for each class = `total_samples / (n_classes × count_of_this_class)`.  
Rare class → small count → larger weight → mistakes are penalized more.

**Expected tradeoff:** Macro F1 should improve. Hamming Loss might slightly worsen (more false positives on rare labels). This tradeoff is a key discussion point.

In [ ]:
# ════════════════════════════════════════════════
#  EXPERIMENT 6 — Class Imbalance Handling
# ════════════════════════════════════════════════
print("Running Experiment 6 — Class Imbalance Handling...\n")

# ── Method A: class_weight='balanced' alone ──────────────────────────────────
lr_balanced = OneVsRestClassifier(
    LogisticRegression(
        C=1.0,
        max_iter=1000,
        random_state=42,
        class_weight='balanced',
        # 'balanced': sklearn automatically computes per-class weights as:
        #   weight[class] = n_samples / (n_classes * count[class])
        # This means rare labels get HIGHER weights → misclassifying them
        # costs more → the model tries harder to get them right.
        # Compare to class_weight=None (Exp 1–5) where all classes equal.
    ),
    n_jobs=-1
)
lr_balanced.fit(X_train_best, Y_train)
Y_pred_bal = lr_balanced.predict(X_val_best)
results_6a = evaluate(Y_val, Y_pred_bal, "Exp 6A: LR + class_weight=balanced")
all_results.append(results_6a)

# ── Method B: balanced weights + optimized thresholds (combining Exp 5 + 6) ──
# Get probability scores from the balanced model
Y_proba_bal = lr_balanced.predict_proba(X_val_best)

# Re-run threshold optimization using balanced model probabilities
best_thresh_bal = []
for j in range(Y.shape[1]):
    label_probs = Y_proba_bal[:, j]
    label_true  = Y_val[:, j]
    best_f1, best_t = 0, 0.5
    for t in thresholds_to_try:
        preds = (label_probs >= t).astype(int)
        f1    = f1_score(label_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    best_thresh_bal.append(round(best_t, 3))

Y_pred_bal_opt = np.zeros_like(Y_proba_bal, dtype=int)
for j, t in enumerate(best_thresh_bal):
    Y_pred_bal_opt[:, j] = (Y_proba_bal[:, j] >= t).astype(int)

results_6b = evaluate(Y_val, Y_pred_bal_opt, "Exp 6B: LR + balanced + opt thresholds")
all_results.append(results_6b)

# ── Macro F1 comparison — the key imbalance metric ───────────────────────────
print("\n── Macro F1 (key metric for rare label performance) ──")
print(f"  Exp 1 (no balancing):             {results_exp1['Macro F1']:.4f}")
print(f"  Exp 6A (balanced weights):        {results_6a['Macro F1']:.4f}")
print(f"  Exp 6B (balanced + opt thresh):   {results_6b['Macro F1']:.4f}")

print("\n── Hamming Loss comparison ──")
print(f"  Exp 1 (no balancing):             {results_exp1['Hamming Loss']:.4f}")
print(f"  Exp 6A (balanced weights):        {results_6a['Hamming Loss']:.4f}")
print(f"  Exp 6B (balanced + opt thresh):   {results_6b['Hamming Loss']:.4f}")

print("\n📝 KEY INSIGHT FOR REPORT:")
print("If Macro F1 improved but Hamming Loss worsened → balancing trades overall accuracy")
print("for better rare-label recall. Discuss this tradeoff explicitly in Section 7.")

---
# SECTION 6 — Full Results Table + Visualizations

In [ ]:
# ── Build the complete results table ──────────────────────────────────────────
results_df = pd.DataFrame(all_results).dropna(subset=['Hamming Loss'])
results_df = results_df.sort_values('Hamming Loss').reset_index(drop=True)

print("\n" + "═"*90)
print("  COMPLETE EXPERIMENT RESULTS  (sorted by Hamming Loss, lowest = best)")
print("═"*90)
print(results_df.to_string(index=False))
print("═"*90)

results_df.to_csv('experiment_results.csv', index=False)
print("\n💾 Saved: experiment_results.csv")

In [ ]:
# ── Comprehensive comparison chart ────────────────────────────────────────────
plot_df = results_df.copy()
short_labels = [
    name.replace('Exp ', 'E').split(':')[-1].strip()[:22]
    for name in plot_df['Experiment']
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
palette = plt.cm.Blues(np.linspace(0.4, 0.85, len(plot_df)))

for ax, metric in zip(axes, ['Hamming Loss', 'Micro F1', 'Macro F1']):
    bars = ax.barh(short_labels, plot_df[metric], color=palette)
    ax.set_xlabel(metric)
    ax.set_title(metric)
    if metric == 'Hamming Loss':
        ax.invert_xaxis()  # flip: best (lowest) reads as 'furthest right'
    # Add value labels on bars
    for bar, val in zip(bars, plot_df[metric]):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=8)

plt.suptitle('All Experiments — Side-by-Side Comparison', fontsize=13)
plt.tight_layout()
plt.savefig('all_experiments_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Saved: all_experiments_comparison.png")

In [ ]:
# ── Per-label F1 breakdown (best model) ──────────────────────────────────────
# Use the experiment with the lowest Hamming Loss as 'best'
best_exp_name = results_df.iloc[0]['Experiment']
print(f"Best experiment: {best_exp_name}")
print("Generating per-label F1 breakdown...\n")

# Use Exp 6B predictions as best (adjust to your actual winner)
best_preds = Y_pred_bal_opt

# f1_score with average=None returns a separate F1 score per label
per_label_f1 = f1_score(Y_val, best_preds, average=None, zero_division=0)

per_label_df = pd.DataFrame({
    'Label':   LABEL_NAMES,
    'F1':      per_label_f1.round(4),
    'Support': Y_val.sum(axis=0).astype(int),
    # Support = how many true-positive instances exist in validation set
}).sort_values('F1', ascending=False)

print(per_label_df.to_string(index=False))

# Plot: color bars green (good) vs red (poor)
colors = ['seagreen' if f >= 0.5 else 'coral' for f in per_label_df['F1']]
plt.figure(figsize=(11, 5))
plt.bar(per_label_df['Label'], per_label_df['F1'], color=colors)
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.7, label='F1=0.5')
plt.title('Per-Label F1 Score — Best Model\n(Green ≥ 0.5  |  Red < 0.5)')
plt.xlabel('SDG 3 Indicator')
plt.ylabel('F1 Score')
plt.ylim(0, 1.05)
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.savefig('per_label_f1.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Saved: per_label_f1.png")

# Connect to Person A's imbalance findings
low_f1_labels = per_label_df[per_label_df['F1'] < 0.4]['Label'].tolist()
print(f"\n⚠ Labels with F1 < 0.4 (hardest to predict): {low_f1_labels}")
print("→ Cross-reference these with Person A's rarest labels. Low support usually = low F1.")

---
# SECTION 7 — Final Test Predictions

Now we apply the best model to the actual test set (`devex_test_clean.csv`).  
The test set has no labels — our predictions are the final deliverable.

**Important:** We retrain on the FULL training data (not just the 80% split) before predicting on test. More training data = better model.

In [ ]:
# ── Step 1: Retrain on full training data (100%, not just 80%) ───────────────
print("Retraining best model on 100% of labeled data...")

# Fit TF-IDF on ALL training text
tfidf_final = TfidfVectorizer(
    max_features=BEST_VOCAB,
    min_df=2,
    max_df=0.95,
    lowercase=False,
    sublinear_tf=True,
    ngram_range=best_ngram,
)
X_all_tfidf  = tfidf_final.fit_transform(X_raw)
# X_raw = all training documents (the full 100%)

# Transform test documents using vocabulary learned from FULL training set
X_test_tfidf = tfidf_final.transform(test_df['clean_text'].values)
print(f"Full training matrix shape: {X_all_tfidf.shape}")
print(f"Test matrix shape:          {X_test_tfidf.shape}")

# ── Step 2: Train final model on ALL labeled data ─────────────────────────────
final_model = OneVsRestClassifier(
    LogisticRegression(
        C=1.0,
        max_iter=1000,
        random_state=42,
        class_weight='balanced',  # keep best config from Experiment 6
    ),
    n_jobs=-1
)
final_model.fit(X_all_tfidf, Y)
# Y = full label matrix (all rows, not just training split)
print("Final model trained on full data. ✅")

# ── Step 3: Predict probabilities on test set ─────────────────────────────────
Y_test_proba = final_model.predict_proba(X_test_tfidf)

# Apply the optimized thresholds from Experiment 6B
Y_test_pred = np.zeros_like(Y_test_proba, dtype=int)
for j, t in enumerate(best_thresh_bal):
    Y_test_pred[:, j] = (Y_test_proba[:, j] >= t).astype(int)

# ── Step 4: Ensure no document has zero labels ────────────────────────────────
# A document with no predicted labels is always wrong.
# For such documents, assign the label with the highest probability.
empty_docs = (Y_test_pred.sum(axis=1) == 0).sum()
for i in range(len(Y_test_pred)):
    if Y_test_pred[i].sum() == 0:
        best_j = np.argmax(Y_test_proba[i])
        # argmax returns the INDEX of the highest value in the array
        Y_test_pred[i, best_j] = 1

print(f"Documents that had 0 labels (corrected): {empty_docs}")
print(f"All documents now have ≥ 1 label: {(Y_test_pred.sum(axis=1) > 0).all()}")

# ── Step 5: Save predictions to CSV ──────────────────────────────────────────
preds_df = pd.DataFrame(Y_test_pred, columns=LABEL_NAMES)

# Include document ID if Person A detected one
id_candidates = [c for c in test_df.columns
                 if 'id' in c.lower() and c not in ['clean_text', RAW_TEXT_COL]]
if id_candidates:
    preds_df.insert(0, id_candidates[0], test_df[id_candidates[0]].values)
    print(f"Included ID column: '{id_candidates[0]}'")

preds_df.to_csv('test_predictions.csv', index=False)
print(f"\n💾 Saved: test_predictions.csv")
print(f"Shape: {preds_df.shape}")
print("\nFirst 5 predictions:")
print(preds_df.head())

In [ ]:
# ── Final notebook summary ────────────────────────────────────────────────────
print("\n" + "█"*65)
print("  PERSON B — NOTEBOOK COMPLETE")
print("█"*65)
print("""
WHAT THIS NOTEBOOK DOES:
  Continues directly from Person A's devex_train_clean.csv and
  devex_test_clean.csv — no reprocessing of raw data.

EXPERIMENTS COMPLETED:
  Exp 1  Baseline: LR + TF-IDF Unigrams
  Exp 2  LR + TF-IDF Bigrams
  Exp 3  Vocabulary size tuning {5k, 10k, 20k, 50k}
  Exp 4  Model comparison: LR vs LinearSVC vs Random Forest
  Exp 5  Threshold optimization (per-label)
  Exp 6A LR + class_weight='balanced'
  Exp 6B LR + balanced + optimized thresholds

OUTPUT FILES:
  experiment_results.csv         ← all experiment metrics
  test_predictions.csv           ← final predictions for test set
  exp3_vocab_tuning.png          ← vocab size vs Hamming Loss
  exp4_model_comparison.png      ← LR vs SVM vs RF comparison
  all_experiments_comparison.png ← all 7 experiments side-by-side
  per_label_f1.png               ← per-SDG-indicator F1 breakdown

HAND OFF TO:
  Person C — receives experiment_results.csv as the baseline to beat
             with advanced embeddings (BERT, Word2Vec, etc.)
  Person D — receives all .png files and experiment_results.csv
             for the Results and Discussion sections of the report
""")